# 🧪 Test Scripts
Notebook tổng hợp các scripts test cho project NutriTrack.

---

### ⚙️ 0. Setup Environment
**Bắt buộc chạy cell này đầu tiên** để thiết lập đường dẫn và load API keys.

In [1]:
import sys
import os
from dotenv import load_dotenv

# 1. Thêm project root vào sys.path để có thể import module 'app'
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"✅ Added to path: {project_root}")

# 2. Load API keys từ .env
env_path = os.path.join(project_root, "config", ".env")
if os.path.exists(env_path):
    load_dotenv(env_path)
    USDA_API_KEY = os.getenv("USDA_API_KEY")
    print(f"✅ .env loaded from: {env_path}")
    print(f"✅ USDA API Key: {USDA_API_KEY[:8]}..." if USDA_API_KEY else "❌ USDA_API_KEY not found in .env")
else:
    print(f"❌ .env file not found at: {env_path}")

✅ Added to path: d:\Project\Code\nutritrack-documentation\app
✅ .env loaded from: d:\Project\Code\nutritrack-documentation\app\config\.env
✅ USDA API Key: pLy0emrT...


## 🍣 Test 1: USDA Nutrition Retrieval
Kiểm tra khả năng kết nối và lấy dữ liệu từ USDA API.

In [ ]:
from third_apis.USDA import USDAClient

# Initialize client
client = USDAClient(api_key=os.getenv("USDA_API_KEY"))


In [4]:
import pytest
from usda_client import USDAClient  # chỉnh lại path nếu cần


# --------------------------------------
# Setup client (mock API key)
# --------------------------------------
client = USDAClient(api_key="DEMO_KEY")


# --------------------------------------
# Test cases
# --------------------------------------
TEST_CASES = [
    # Basic
    ("rice (gạo)", "rice"),
    ("fried rice (cơm chiên)", "fried rice"),
    ("Grilled pork (thịt nướng)", "grilled pork"),
    ("pho (vietnamese noodle soup)", "pho"),
    ("soup (canh)", "soup"),

    # Short prefix → merge
    ("bò (beef)", "bo beef"),
    ("gà (chicken)", "ga chicken"),
    ("heo (pork)", "heo pork"),
    ("cá (fish)", "ca fish"),

    # No parentheses
    ("rice", "rice"),
    ("  Fried Rice  ", "fried rice"),
    ("GRILLED PORK!!!", "grilled pork"),
    ("Beef-steak", "beefsteak"),

    # Spacing & punctuation
    ("  rice   ( gạo )  ", "rice"),
    ("fried-rice (cơm chiên)!!!", "friedrice"),
    ("chicken, grilled (gà nướng)", "chicken grilled"),

    # Edge cases
    ("", ""),
    ("   ", ""),
    ("(beef)", "beef"),
    ("() rice", "rice"),
    ("rice ()", "rice"),
]


# --------------------------------------
# Pretty test output
# --------------------------------------
@pytest.mark.parametrize("input_text, expected", TEST_CASES)
def test_normalize_query(input_text, expected):
    output = client._normalize_query(input_text)

    if output == expected:
        print(f"✓ PASS  | '{input_text}' → '{output}'")
    else:
        print(f"✗ FAIL  | '{input_text}' → '{output}' (expected '{expected}')")

    assert output == expected


# --------------------------------------
# Optional: Summary after all tests
# --------------------------------------
def test_summary():
    passed = 0
    failed = 0

    for input_text, expected in TEST_CASES:
        output = client._normalize_query(input_text)
        if output == expected:
            passed += 1
        else:
            failed += 1

    print("\n==============================")
    print(f"Total tests : {len(TEST_CASES)}")
    print(f"✓ Passed     : {passed}")
    print(f"✗ Failed     : {failed}")
    print("==============================\n")

    assert failed == 0

ModuleNotFoundError: No module named 'pytest'

In [3]:
all_test_cases = [
    # Basic
    ("rice (gạo)", "rice"),
    ("fried rice (cơm chiên)", "fried rice"),
    ("Grilled pork (thịt nướng)", "grilled pork"),
    ("pho (vietnamese noodle soup)", "pho"),
    ("soup (canh)", "soup"),

    # Short prefix
    ("bò (beef)", "bo beef"),
    ("gà (chicken)", "ga chicken"),
    ("heo (pork)", "heo pork"),
    ("cá (fish)", "ca fish"),

    # No parentheses
    ("rice", "rice"),
    ("  Fried Rice  ", "fried rice"),
    ("GRILLED PORK!!!", "grilled pork"),
    ("Beef-steak", "beefsteak"),

    # Spacing & punctuation
    ("  rice   ( gạo )  ", "rice"),
    ("fried-rice (cơm chiên)!!!", "friedrice"),
    ("chicken, grilled (gà nướng)", "chicken grilled"),

    # Edge cases
    ("", ""),
    ("   ", ""),
    ("(beef)", "beef"),
    ("() rice", "rice"),
    ("rice ()", "rice"),
]

for inp, expected in all_test_cases:
    out = client._normalize_query(inp)
    print(f"{inp} → {out} | OK: {out == expected}")

rice (gạo) → rice | OK: True
fried rice (cơm chiên) → fried rice | OK: True
Grilled pork (thịt nướng) → grilled pork | OK: True
pho (vietnamese noodle soup) → pho | OK: True
soup (canh) → soup | OK: True
bò (beef) → bò | OK: False
gà (chicken) → gà | OK: False
heo (pork) → heo | OK: False
cá (fish) → cá | OK: False
rice → rice | OK: True
  Fried Rice   → fried rice | OK: True
GRILLED PORK!!! → grilled pork | OK: True
Beef-steak → beefsteak | OK: True
  rice   ( gạo )   → rice | OK: True
fried-rice (cơm chiên)!!! → friedrice | OK: True
chicken, grilled (gà nướng) → chicken grilled | OK: True
 →  | OK: True
    →  | OK: True
(beef) →  | OK: False
() rice → rice | OK: True
rice () → rice | OK: True


## 📸 Test 2: Pipeline - Qwen3VL analyze_food ➡️ USDA Nutrition Calculation

**Mục tiêu:**
1. Dùng `Qwen3VL.analyze_food()` phát hiện các món ăn và ingredients.
2. **Làm gọn logic:** Tính toán và lưu kết quả vào biến `final_nutrition_results` để tái sử dụng làm API.
3. **Tách biệt hiển thị:** Dùng cell riêng để render kết quả.

In [3]:
from models.QWEN3VL import Qwen3VL
import os

# Initialize Qwen3VL
qwen = Qwen3VL()

✅ Qwen3 VL Ready! (model: qwen.qwen3-vl-235b-a22b, region: us-east-1)


In [4]:
# Path ảnh test
image_path = r"D:\Project\Code\nutritrack-documentation\data\images\food\fast_food.jpg"
print(f"📸 Phân tích ảnh: {image_path}")
food_list = qwen.analyze_food(image_path)
print(f"\n✅ Phát hiện {len(food_list.items)} món ăn.")

📸 Phân tích ảnh: D:\Project\Code\nutritrack-documentation\data\images\food\fast_food.jpg
🔄 Loading image from D:\Project\Code\nutritrack-documentation\data\images\food\fast_food.jpg...
🚀 Analyzing with 'qwen.qwen3-vl-235b-a22b' → FoodList...

✅ Phát hiện 0 món ăn.


In [6]:
food_list.model_dump()

{'items': [{'name': 'Fried Potatoes',
   'vi_name': 'Khoai tây chiên',
   'ingredients': [{'name': 'Potatoes',
     'vi_name': 'Khoai tây',
     'estimated_weight_g': 200.0,
     'confidence': 0.95},
    {'name': 'Oil',
     'vi_name': 'Dầu ăn',
     'estimated_weight_g': 15.0,
     'confidence': 0.85},
    {'name': 'Salt',
     'vi_name': 'Muối',
     'estimated_weight_g': 2.0,
     'confidence': 0.9}],
   'total_estimated_calories': 250.0},
  {'name': 'Fried Chicken Burger',
   'vi_name': 'Burger gà rán',
   'ingredients': [{'name': 'Chicken Patty',
     'vi_name': 'Thịt gà rán',
     'estimated_weight_g': 120.0,
     'confidence': 0.9},
    {'name': 'Bun',
     'vi_name': 'Bánh mì hamburger',
     'estimated_weight_g': 80.0,
     'confidence': 0.95},
    {'name': 'Lettuce',
     'vi_name': 'Rau diếp',
     'estimated_weight_g': 10.0,
     'confidence': 0.8},
    {'name': 'Onion',
     'vi_name': 'Hành tây',
     'estimated_weight_g': 10.0,
     'confidence': 0.85},
    {'name': 'May

In [7]:
import time

# ⚙️ CALCULATION CELL: Xử lý logic và lưu vào biến 'final_nutrition_results'
# Biến này có thể dùng làm Response cho API sau này
final_nutrition_results = []

print("🔄 Đang tính toán dinh dưỡng thực tế từ USDA...")

for food in food_list.items:
    food_data = {
        "name": food.name,
        "vi_name": food.vi_name,
        "qwen_estimated_calories": food.total_estimated_calories,
        "ingredients": []
    }
    
    total_nutrition = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0}
    
    for ing in food.ingredients:
        # 1. Tra cứu USDA (luôn trả về per 100g)
        usda_100g = client.get_nutrition(ing.name)
        
        # 2. Tính toán dựa trên trọng lượng thực
        weight = ing.estimated_weight_g if ing.estimated_weight_g else 0
        ratio = weight / 100.0
        
        nutrition = {
            "calories": usda_100g['calories'] * ratio,
            "protein": usda_100g['protein'] * ratio,
            "carbs": usda_100g['carbs'] * ratio,
            "fat": usda_100g['fat'] * ratio
        }
        
        # 3. Lưu thông tin ingredient
        food_data["ingredients"].append({
            "name": ing.name,
            "vi_name": ing.vi_name,
            "weight_g": weight,
            "usda_100g": usda_100g,
            "nutrition": nutrition
        })
        
        # Cộng dồn tổng
        for k in total_nutrition: total_nutrition[k] += nutrition[k]
        
        time.sleep(0.2) # Rate limit
    
    food_data["total_nutrition"] = total_nutrition
    avg_calories = (total_nutrition["calories"] + food.total_estimated_calories) / 2
    food_data["average_calories"] = avg_calories
    final_nutrition_results.append(food_data)

print("✅ Tính toán xong. Dữ liệu đã sẵn sàng trong biến 'final_nutrition_results'.")

🔄 Đang tính toán dinh dưỡng thực tế từ USDA...
{'fdcId': 576920, 'description': 'POTATOES', 'dataType': 'Branded', 'gtinUpc': '629307123344', 'publishedDate': '2019-04-01', 'brandOwner': 'THE LITTLE POTATO COMPANY', 'ingredients': '', 'marketCountry': 'United States', 'foodCategory': 'Pre-Packaged Fruit & Vegetables', 'modifiedDate': '2018-03-02', 'dataSource': 'LI', 'servingSizeUnit': 'g', 'servingSize': 148.0, 'householdServingFullText': '5 TO 6 POTATOES', 'tradeChannels': ['NO_TRADE_CHANNEL'], 'allHighlightFields': '<b>Brand Owner</b>: THE LITTLE <em>POTATO</em> COMPANY', 'score': 904.02594, 'microbes': [], 'foodNutrients': [{'nutrientId': 1004, 'nutrientName': 'Total lipid (fat)', 'nutrientNumber': '204', 'unitName': 'G', 'derivationCode': 'LCCD', 'derivationDescription': 'Calculated from a daily value percentage per serving size measure', 'derivationId': 75, 'value': 0.0, 'foodNutrientSourceId': 9, 'foodNutrientSourceCode': '12', 'foodNutrientSourceDescription': "Manufacturer's an

In [8]:
final_nutrition_results

[{'name': 'Fried Potatoes',
  'vi_name': 'Khoai tây chiên',
  'qwen_estimated_calories': 250.0,
  'ingredients': [{'name': 'Potatoes',
    'vi_name': 'Khoai tây',
    'weight_g': 200.0,
    'usda_100g': {'calories': 74.0, 'protein': 2.7, 'fat': 0.0, 'carbs': 16.9},
    'nutrition': {'calories': 148.0,
     'protein': 5.4,
     'carbs': 33.8,
     'fat': 0.0}},
   {'name': 'Oil',
    'vi_name': 'Dầu ăn',
    'weight_g': 15.0,
    'usda_100g': {'calories': 867, 'protein': 0.0, 'fat': 93.3, 'carbs': 0.0},
    'nutrition': {'calories': 130.04999999999998,
     'protein': 0.0,
     'carbs': 0.0,
     'fat': 13.995}},
   {'name': 'Salt',
    'vi_name': 'Muối',
    'weight_g': 2.0,
    'usda_100g': {'calories': 0.0, 'protein': 0.0, 'fat': 0.0, 'carbs': 0.0},
    'nutrition': {'calories': 0.0, 'protein': 0.0, 'carbs': 0.0, 'fat': 0.0}}],
  'total_nutrition': {'calories': 278.04999999999995,
   'protein': 5.4,
   'carbs': 33.8,
   'fat': 13.995},
  'average_calories': 264.025},
 {'name': 'Fried

In [9]:
# 📊 DISPLAY CELL: Hiển thị báo cáo từ 'final_nutrition_results'
print(f"{'='*95}")
print(f"📊 BÁO CÁO DINH DƯỠNG CHI TIẾT")
print(f"{'='*95}\n")

for food in final_nutrition_results:
    print(f"🍽️  MÓN: {food['name']} ({food['vi_name']})")
    print(f"🔥 Calories (AI ước tính): {food['qwen_estimated_calories']} kcal")
    print(f"{'Ingredient':<20} | {'Weight':>8} | {'USDA(100g) Cal/P/C/F':<24} | {'REAL Cal/P/C/F':<24}")
    print("-" * 95)
    
    for ing in food['ingredients']:
        u = ing['usda_100g']
        r = ing['nutrition']
        
        u_str = f"{u['calories']:>5.0f}/{u['protein']:>4.1f}/{u['carbs']:>4.1f}/{u['fat']:>4.1f}"
        r_str = f"{r['calories']:>5.0f}/{r['protein']:>4.1f}/{r['carbs']:>4.1f}/{r['fat']:>4.1f}"
        
        print(f"   {ing['name']:<17} | {ing['weight_g']:>7.1f}g | {u_str:<24} | {r_str:<24}")
    
    t = food['total_nutrition']
    avg_calories = food["average_calories"]
    print("-" * 95)
    print(f"📊 TỔNG CỘNG THỰC TẾ: {'':<10} | Avg Calories: {avg_calories:>4.1f}g | Cal: {t['calories']:>5.0f} | Pro: {t['protein']:>4.1f}g | Carb: {t['carbs']:>4.1f}g | Fat: {t['fat']:>4.1f}g")
    print("\n" + "="*95 + "\n")

📊 BÁO CÁO DINH DƯỠNG CHI TIẾT

🍽️  MÓN: Fried Potatoes (Khoai tây chiên)
🔥 Calories (AI ước tính): 250.0 kcal
Ingredient           |   Weight | USDA(100g) Cal/P/C/F     | REAL Cal/P/C/F          
-----------------------------------------------------------------------------------------------
   Potatoes          |   200.0g |    74/ 2.7/16.9/ 0.0     |   148/ 5.4/33.8/ 0.0    
   Oil               |    15.0g |   867/ 0.0/ 0.0/93.3     |   130/ 0.0/ 0.0/14.0    
   Salt              |     2.0g |     0/ 0.0/ 0.0/ 0.0     |     0/ 0.0/ 0.0/ 0.0    
-----------------------------------------------------------------------------------------------
📊 TỔNG CỘNG THỰC TẾ:            | Avg Calories: 264.0g | Cal:   278 | Pro:  5.4g | Carb: 33.8g | Fat: 14.0g


🍽️  MÓN: Fried Chicken Burger (Burger gà rán)
🔥 Calories (AI ước tính): 500.0 kcal
Ingredient           |   Weight | USDA(100g) Cal/P/C/F     | REAL Cal/P/C/F          
---------------------------------------------------------------------------

## 🔬 Test 3: So sánh `converse` vs `instructor chat.completions`

**Mục tiêu:** So sánh 2 phương pháp gọi Qwen3-VL:
1. **Method 1 (Converse):** `client.converse()` + manual JSON parse + Pydantic validate
2. **Method 2 (Instructor):** `instructor.from_provider()` + `chat.completions.create()` + auto Pydantic validate

Cùng 1 ảnh, cùng 1 prompt, so sánh kết quả và thời gian.

In [18]:
import time
from models.QWEN3VL import Qwen3VL, FoodList

# Initialize
qwen = Qwen3VL()

# Test image
test_image = r"D:\Project\Code\nutritrack-documentation\data\images\food\com_tam.jpg"

# ─── Method 1: Converse API ──────────────────────────────────────────
print("=" * 60)
print("📡 METHOD 1: client.converse() + manual Pydantic")
print("=" * 60)

start = time.time()
result_converse = qwen.analyze_food(test_image)
time_converse = time.time() - start

print(f"\n⏱️ Thời gian: {time_converse:.2f}s")
print(f"📊 Số món: {len(result_converse.items)}")
for item in result_converse.items:
    print(f"   🍽️ {item.name} ({item.vi_name}) - {item.total_estimated_calories} kcal")
    print(f"      Ingredients: {len(item.ingredients)} loại")

✅ Qwen3 VL Ready! (model: qwen.qwen3-vl-235b-a22b, region: us-east-1)
📡 METHOD 1: client.converse() + manual Pydantic
🔄 Loading image from D:\Project\Code\nutritrack-documentation\data\images\food\com_tam.jpg...
🚀 Analyzing with 'qwen.qwen3-vl-235b-a22b' → FoodList...

⏱️ Thời gian: 11.59s
📊 Số món: 1
   🍽️ Vietnamese Pork Rib Rice Plate (Cơm Sườn Nướng) - 780.0 kcal
      Ingredients: 12 loại


In [19]:
# ─── Method 2: Instructor + chat.completions ────────────────────────
print("=" * 60)
print("🤖 METHOD 2: instructor.from_provider() + chat.completions")
print("=" * 60)

start = time.time()
result_instructor = qwen.analyze_food_with_instructor(test_image)
time_instructor = time.time() - start

print(f"\n⏱️ Thời gian: {time_instructor:.2f}s")
print(f"📊 Số món: {len(result_instructor.items)}")
for item in result_instructor.items:
    print(f"   🍽️ {item.name} ({item.vi_name}) - {item.total_estimated_calories} kcal")
    print(f"      Ingredients: {len(item.ingredients)} loại")

🤖 METHOD 2: instructor.from_provider() + chat.completions


AttributeError: 'Qwen3VL' object has no attribute 'analyze_food_with_instructor'

In [ ]:
# ─── So sánh kết quả ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📋 SO SÁNH KẾT QUẢ")
print("=" * 60)
print(f"{'Tiêu chí':<30} | {'Converse':>12} | {'Instructor':>12}")
print("-" * 60)
print(f"{'Thời gian (s)':<30} | {time_converse:>11.2f}s | {time_instructor:>11.2f}s")
print(f"{'Số món phát hiện':<30} | {len(result_converse.items):>12} | {len(result_instructor.items):>12}")

# So sánh calories
cal_converse = sum(i.total_estimated_calories or 0 for i in result_converse.items)
cal_instructor = sum(i.total_estimated_calories or 0 for i in result_instructor.items)
print(f"{'Tổng calories ước tính':<30} | {cal_converse:>11.0f}  | {cal_instructor:>11.0f} ")

# So sánh ingredients
ing_converse = sum(len(i.ingredients) for i in result_converse.items)
ing_instructor = sum(len(i.ingredients) for i in result_instructor.items)
print(f"{'Tổng số ingredients':<30} | {ing_converse:>12} | {ing_instructor:>12}")
print("-" * 60)

faster = 'Converse' if time_converse < time_instructor else 'Instructor'
print(f"\n🏆 {faster} nhanh hơn!")

In [ ]:
# Xem raw data chi tiết
print("\n📦 Converse result:")
print(result_converse.model_dump())
print("\n📦 Instructor result:")
print(result_instructor.model_dump())

call tool function